<a href="https://colab.research.google.com/github/helmernet/Helmernet/blob/main/Copia_de_modificacion_vd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""Video_naturaleza.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1Nbqva440rJVMyOVke37R7IvPa-PNSNvY
"""

!pip install gtts moviepy imageio-ffmpeg

import os
from moviepy.editor import *
from gtts import gTTS
from google.colab import files

# Función para convertir texto a voz
def text_to_speech(text, lang='es', output_file="temp_audio.mp3"):
    tts = gTTS(text=text, lang=lang)
    tts.save(output_file)
    return output_file

# Función para crear un video con imagen/video, audio y música de fondo
def create_video_with_music(text, media_path="static_image.jpg", audio_file="temp_audio.mp3", music_file=None, output_video="output_video.mp4", duration=5):
    # Determinar si el archivo de entrada es una imagen o un video
    if media_path.lower().endswith(('.mp4', '.mov', '.avi', '.mkv')):
        # Es un video
        media_clip = VideoFileClip(media_path).subclip(0, duration)
    else:
        # Es una imagen
        media_clip = ImageClip(media_path, duration=duration)

    # Ajustar tamaño
    media_clip = media_clip.resize(height=720)
    media_clip = media_clip.set_position(("center", "center"))

    # Cargar el archivo de audio (voz)
    audio_clip = AudioFileClip(audio_file)

    # Combinar media y audio
    video_clip = media_clip.set_audio(audio_clip)

    # Si se proporciona música de fondo, mezclarla con el audio
    if music_file:
        music_clip = AudioFileClip(music_file)
        music_clip = music_clip.subclip(0, duration)
        music_clip = music_clip.volumex(0.2)

        # Mezclar el audio de voz con la música de fondo
        final_audio = CompositeAudioClip([audio_clip, music_clip])
        video_clip = video_clip.set_audio(final_audio)

    # Guardar el video final
    video_clip.write_videofile(output_video, fps=24, codec="libx264")
    return output_video

# Función principal para generar el video y permitir su descarga
def generate_and_download_video():
    # Solicitar entrada del usuario
    input_text = input("Ingresa el texto para el video: ")
    lang = input("Selecciona el idioma (es/en): ") or "es"
    duration = int(input("Ingresa la duración del video (segundos): ") or 5)

    # Preguntar si el usuario quiere subir una imagen o un video
    print("¿Qué deseas subir como fondo?")
    media_type = input("Escribe '1' para imagen o '2' para video: ")

    if media_type == '2':
        print("Sube un video MP4 como fondo.")
        uploaded_media = files.upload()
        if uploaded_media:
            media_path = list(uploaded_media.keys())[0]
        else:
            # Usar un video predeterminado si no se sube ninguno
            print("No se subió ningún video. Usando imagen predeterminada.")
            image_url = "https://via.placeholder.com/1280x720.png?text=Sample+Image"
            !wget -O static_image.jpg -q {image_url}
            media_path = "static_image.jpg"
    else:
        print("Sube una imagen personalizada o usa una imagen predeterminada.")
        uploaded_media = files.upload()
        if uploaded_media:
            media_path = list(uploaded_media.keys())[0]
        else:
            # Descargar una imagen predeterminada si no se sube ninguna
            image_url = "https://via.placeholder.com/1280x720.png?text=Sample+Image"
            !wget -O static_image.jpg -q {image_url}
            media_path = "static_image.jpg"

    # Permitir al usuario subir un archivo de música de fondo
    print("Sube un archivo de música de fondo (opcional).")
    uploaded_music = files.upload()
    music_file = list(uploaded_music.keys())[0] if uploaded_music else None

    # Convertir texto a voz
    audio_file = text_to_speech(input_text, lang=lang)

    # Crear el video
    output_video = create_video_with_music(
        input_text,
        media_path=media_path,
        audio_file=audio_file,
        music_file=music_file,
        duration=duration
    )

    # Permitir la descarga del video
    files.download(output_video)

    # Limpieza de archivos temporales
    if os.path.exists("temp_audio.mp3"):
        os.remove("temp_audio.mp3")
    if not uploaded_media and os.path.exists("static_image.jpg"):
        os.remove("static_image.jpg")

# Ejecutar la función principal
if __name__ == "__main__":
    generate_and_download_video()